<a href="https://colab.research.google.com/github/Ans365332/6may-file-example/blob/main/Webhook_and_Queue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Listen to Queue and Webhooks

In [2]:
!pip install -qU langchain langchain-google-genai google-generativeai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [3]:
# LLM Setup

In [4]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your  API Key:")


from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model= "gemini-2.5-flash",temperature=0)



Enter your  API Key:··········


In [5]:
import json
import time
import uuid
import random
import queue
import hashlib
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable, List , Dict , Any , Optional

In [6]:
# Target-1 ::  Listen to Webhook and Queues

In [8]:
class SimulatedEventQueue:

  def __init__(self):
    self._queue: "queue.Queue" = queue.Queue()

  def push_event(self, event_type:str, payload:Dict[str,Any], event_id:Optional[str]=None):
    event = {
        "event_id": event_id or str(uuid.uuid4()),
        "event_type": event_type,
        "payload": payload,
        "recieved_at": datetime.now().strftime("%Y-%M-%D  %H:%M:%S")

    }
    self.queue.put(event)
    print(f"[listener] event received: type='{event_type}' id={event['event_id'][:8]}...")
    return event

  def listen_and_drain(self):
    events = []
    while not self._queue.empty():
      events.append(self._queue.get())
    return events

  def is_empty(self):
    return self._queue.empty()

event_queue = SimulatedEventQueue()
print("Queue Set Up is ready.....")


Queue Set Up is ready.....


In [ ]:
#### Execute Workflow on trigger

In [9]:
class WorkflowRegistry:

  def __init__(self):
    self._workflows: Dict[str , Callable] = {}

  def register(self, event_type:str , workflow_fn: Callable):
    self._workflows[event_type] = workflow_fn
    print(f"[Workflow-registery] registered workflow for event_type = '{event_type}' ")

  def get(self, event_type: str):
    return self._workflows.get(event_type)

  def known_event_types(self):
    return list(self._workflows.keys())

def smart_route_event_type(event_type: str , payload: Dict[str,Any], registry: "WorkflowRegistry"):
  known = registry.known_event_types()
  prompt = f'''A system recieved an event with type "{event_type}" and payload {json.dumps(payload)}.
  known workflow event type are: {known}

  which known event type is this closest to? Respond with only the exact matching
  string from the known list, or "None" if nothing fits.'''

  guess = llm.invoke(prompt).content.strip()
  if guess in known:
    print(f"[workflow-registry] Gemini smart-routed unknown type '{event_type}' -> '{guess}'")
    return guess
  print(f"[workflow-registry] no matching workflow found for '{event_type}' (Gemini said: '{guess}')")
  return None


# --- Demo Workflows -----

def workflow_send_order_confirmation(payload: Dict[str,Any]):
  order_id = payload.get("order_id","unknown")
  return f"Confirmation email sent for order #{order_id}"

def workflow_notify_payment_failure(payload: Dict[str,Any]):
  user = payload.get("user","unknown")
  if random.ramdom() < 0.6:
    raise RuntimeError("Payment gateway notification service timeout")

  return f"Payment failure notification sent to {user}"


def workflow_send_welcome_message(payload: Dict[str,Any]):
  name = payload.get("name","user")
  return f"Welcome message sent to {name}"


workflow_registry = WorkflowRegistry()
workflow_registry.register("Order.created", workflow_send_order_confirmation)
workflow_registry.register("Payment.failed", workflow_notify_payment_failure)
workflow_registry.register("User.signup", workflow_send_welcome_message)

print("\nRegistered workflows:",workflow_registry.known_event_types())


[Workflow-registery] registered workflow for event_type = 'Order.created' 
[Workflow-registery] registered workflow for event_type = 'Payment.failed' 
[Workflow-registery] registered workflow for event_type = 'User.signup' 

Registered workflows: ['Order.created', 'Payment.failed', 'User.signup']


In [ ]:
# part-3 ::: Idempotent Execution -> store result in cache of already processed

In [10]:
class IdempotencyStore:

  def __init__(self):
    self._processed: Dict[str, Dict[str,Any]] = {}

  def already_processed(self, event_id:str):
    return event_id in self._processed

  def get_cached_result(self, event_id: str): #-> Optional[Dic[str,Any]]
    return self._processed.get(event_id)

  def mark_processed(self, event_id: str , result: Dict[str,Any]):
    self._processed[event_id] = result

  def stats(self):#-> Dict[str,int]
     return {"total_unique_event_processed": len(self._processed)}

idempotency_store = IdempotencyStore()
print("IdempotencyStore ready")

IdempotencyStore ready


In [ ]:
#Part-4 ::: Dead Letter Handling and Retry Logic

In [11]:
MAX_RETRIES = 3
BASE_BACKOFF_SECONDS = 0.5

class DeadLetterQueue:

  def __init__(self):
    self.items: List[Dict[str,Any]] = []

  def add(self, event: Dict[str,Any], error: str, attempts: int):
    entry = {
        "event": event,
        "error":error,
        "attempts":attempts,
        "failed_at":datetime.now().strftime("%Y-%M-%D %H:%M:%S"),
    }

    self.items.append(entry)
    print(f"[DLQ] event_id={event['event_id'][:8]} .....moved to Dead Letter Queue"
          f"ater {attempts} attempts. Reason: {error}")

    def list_items(self): # -> List[Dict[str,Any]]
      return self.items

    def remove(self, event_id: str):
      self.itema = [i for i in self.items if i["event"]["event_id"] != event_id]

dlq = DeadLetterQueue()


def execute_with_retry(workflow_fn: Callable, event: Dict[str,Any], dlq: DeadLetterQueue,
                       max_retries: int = MAX_RETRIES):

  """
  -> Dead-letter Handling + Retry Logic <-
  Exponential backoff ke sath retry karta hai; sab attempta fail hone par DLQ me dal deta hai.
  """

  last_error = None
  for attempts in range(1, max_retries+1):
    try:
      result_text = workflow_fn(event["payload"])
      print(f"[retry] attempt {attempts}/{max_retries} succeded for event_id = {event['event_id'][:8]}....")
      return {"status": "success", "output": result_text,"attempts":attempts}

    except Exception as e:
      last_error = str(e)
      wait_time = BASE_BACKOFF_SECONDS * (2**(attempts - 1))  #exponential backoff: 0.5,1.0,2.0.....
      print(f"[retry] attempts {attempts}/{max_retries} failed: '{last_error}"
            f"-> backing off {wait_time}s before next try")


    # Sare retries fali ho gye -> DLQ me bhejo
    dlq.add(event,last_error,max_retries)
    return {"status": "dead_lettered","error": last_error, "attempts": max_retries}


print("Retry+Dead Letter Queue ready")



Retry+Dead Letter Queue ready
